In [1]:
# ==========================================
# CREDIT RISK & LENDING ML PROJECT
# credit_risk_model.py
# ==========================================

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)

from sklearn.ensemble import IsolationForest

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv("credit_applicants.csv")

print("=" * 50)
print("FIRST 5 ROWS")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nDataset Information")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

# ==========================================
# EDA
# ==========================================

default_rate = df["default"].mean() * 100

missing_bureau = (
    df["credit_bureau_score"].isna().mean() * 100
)

print("\nDefault Rate : {:.2f}%".format(default_rate))
print("Missing Bureau Score : {:.2f}%".format(missing_bureau))

# Thin File Feature
df["is_thin_file"] = df["credit_bureau_score"].isna().astype(int)

print("\nThin File Applicants")
print(df["is_thin_file"].value_counts())

# ==========================================
# Features & Target
# ==========================================

X = df.drop(["applicant_id", "default"], axis=1)

y = df["default"]

# ==========================================
# Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

FIRST 5 ROWS
  applicant_id  age  monthly_income_inr  existing_loans_count  \
0      APP1000   59               41646                     4   
1      APP1001   49              119185                     0   
2      APP1002   35               38049                     0   
3      APP1003   28              113116                     0   
4      APP1004   41              112379                     2   

   credit_utilization_ratio  upi_monthly_inflow_inr  bounced_payments_count  \
0                      0.80                   17398                       2   
1                      0.08                   58503                       3   
2                      0.59                    8638                       1   
3                      0.26                    8570                       2   
4                      0.16                   70785                       3   

   credit_bureau_score employment_type  default  
0                  NaN        salaried        1  
1                380.

In [2]:
# ==========================================
# PREPROCESSING
# ==========================================

# -----------------------------
# Median Imputation
# (Training data only)
# -----------------------------

median_bureau = X_train["credit_bureau_score"].median()

print("\nTraining Median Bureau Score:", median_bureau)

X_train["credit_bureau_score"] = X_train["credit_bureau_score"].fillna(
    median_bureau
)

X_test["credit_bureau_score"] = X_test["credit_bureau_score"].fillna(
    median_bureau
)

# -----------------------------
# One-Hot Encoding
# -----------------------------

X_train = pd.get_dummies(
    X_train,
    columns=["employment_type"],
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=["employment_type"],
    drop_first=True
)

# Make sure both datasets have identical columns
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

# -----------------------------
# Standard Scaling
# -----------------------------

scaler = StandardScaler()

numeric_columns = [
    "age",
    "monthly_income_inr",
    "existing_loans_count",
    "credit_utilization_ratio",
    "upi_monthly_inflow_inr",
    "bounced_payments_count",
    "credit_bureau_score",
    "is_thin_file"
]

X_train[numeric_columns] = scaler.fit_transform(
    X_train[numeric_columns]
)

X_test[numeric_columns] = scaler.transform(
    X_test[numeric_columns]
)

print("\nPreprocessing Completed Successfully")
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape :", X_test.shape)


Training Median Bureau Score: 612.0

Preprocessing Completed Successfully
Training Data Shape: (300, 10)
Testing Data Shape : (100, 10)


In [3]:
# ==========================================
# LOGISTIC REGRESSION MODEL
# ==========================================

lr_model = LogisticRegression(random_state=42)

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_precision = precision_score(y_test, lr_pred)
lr_recall = recall_score(y_test, lr_pred)
lr_f1 = f1_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_prob)

print("\n" + "="*50)
print("LOGISTIC REGRESSION RESULTS")
print("="*50)

print(f"Accuracy : {lr_accuracy:.4f}")
print(f"Precision: {lr_precision:.4f}")
print(f"Recall   : {lr_recall:.4f}")
print(f"F1 Score : {lr_f1:.4f}")
print(f"ROC AUC  : {lr_auc:.4f}")

# Confusion Matrix
cm_lr = confusion_matrix(y_test, lr_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_lr,
    display_labels=["No Default", "Default"]
)

disp.plot()
plt.title("Logistic Regression Confusion Matrix")
plt.savefig("confusion_matrix_lr.png")
plt.close()


# ==========================================
# DECISION TREE MODEL
# ==========================================

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)
dt_prob = dt_model.predict_proba(X_test)[:, 1]

dt_accuracy = accuracy_score(y_test, dt_pred)
dt_precision = precision_score(y_test, dt_pred)
dt_recall = recall_score(y_test, dt_pred)
dt_f1 = f1_score(y_test, dt_pred)
dt_auc = roc_auc_score(y_test, dt_prob)

print("\n" + "="*50)
print("DECISION TREE RESULTS")
print("="*50)

print(f"Accuracy : {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall   : {dt_recall:.4f}")
print(f"F1 Score : {dt_f1:.4f}")
print(f"ROC AUC  : {dt_auc:.4f}")

# Confusion Matrix
cm_dt = confusion_matrix(y_test, dt_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_dt,
    display_labels=["No Default", "Default"]
)

disp.plot()
plt.title("Decision Tree Confusion Matrix")
plt.savefig("confusion_matrix_dt.png")
plt.close()


# ==========================================
# ROC CURVE
# ==========================================

lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_prob)

plt.figure(figsize=(8,6))

plt.plot(
    lr_fpr,
    lr_tpr,
    label=f"Logistic Regression (AUC={lr_auc:.3f})"
)

plt.plot(
    dt_fpr,
    dt_tpr,
    label=f"Decision Tree (AUC={dt_auc:.3f})"
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()

plt.savefig("roc_curve.png")
plt.close()


# ==========================================
# MODEL COMPARISON TABLE
# ==========================================

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree"
    ],
    "Accuracy": [
        lr_accuracy,
        dt_accuracy
    ],
    "Precision": [
        lr_precision,
        dt_precision
    ],
    "Recall": [
        lr_recall,
        dt_recall
    ],
    "F1 Score": [
        lr_f1,
        dt_f1
    ],
    "ROC AUC": [
        lr_auc,
        dt_auc
    ]
})

print("\n")
print("="*70)
print("MODEL COMPARISON")
print("="*70)

print(comparison)


LOGISTIC REGRESSION RESULTS
Accuracy : 0.7600
Precision: 0.3889
Recall   : 0.3500
F1 Score : 0.3684
ROC AUC  : 0.7188

DECISION TREE RESULTS
Accuracy : 0.6500
Precision: 0.2222
Recall   : 0.3000
F1 Score : 0.2553
ROC AUC  : 0.5188


MODEL COMPARISON
                 Model  Accuracy  Precision  Recall  F1 Score  ROC AUC
0  Logistic Regression      0.76   0.388889    0.35  0.368421  0.71875
1        Decision Tree      0.65   0.222222    0.30  0.255319  0.51875


In [4]:
# ==========================================
# RISK-BASED PRICING
# ==========================================

risk_df = pd.DataFrame({
    "Actual_Default": y_test.values,
    "Predicted_Probability": lr_prob
})

# Create 4 risk tiers using quartiles
risk_df["Risk_Tier"] = pd.qcut(
    risk_df["Predicted_Probability"],
    q=4,
    labels=[
        "Low Risk",
        "Medium Risk",
        "High Risk",
        "Very High Risk"
    ]
)

interest_rates = {
    "Low Risk": "10% - 12%",
    "Medium Risk": "13% - 16%",
    "High Risk": "17% - 20%",
    "Very High Risk": "21% - 28%"
}

pricing_table = (
    risk_df
    .groupby("Risk_Tier", observed=True)
    .agg(
        Applicants=("Risk_Tier", "count"),
        Avg_Default_Probability=("Predicted_Probability", "mean"),
        Observed_Default_Rate=("Actual_Default", "mean")
    )
    .reset_index()
)

pricing_table["Observed_Default_Rate"] = (
    pricing_table["Observed_Default_Rate"] * 100
).round(2)

pricing_table["Avg_Default_Probability"] = (
    pricing_table["Avg_Default_Probability"] * 100
).round(2)

pricing_table["Interest_Rate"] = (
    pricing_table["Risk_Tier"].map(interest_rates)
)

print("\n")
print("="*70)
print("RISK-BASED PRICING TABLE")
print("="*70)

print(pricing_table)

pricing_table.to_csv(
    "risk_pricing_table.csv",
    index=False
)

# ==========================================
# ISOLATION FOREST
# ==========================================

behaviour = pd.read_csv("txn_behaviour.csv")

print("\nTransaction Dataset Shape")
print(behaviour.shape)

features = behaviour[
    [
        "txn_hour",
        "is_new_device",
        "txn_amount_inr"
    ]
]

scaler_iso = StandardScaler()

features_scaled = scaler_iso.fit_transform(features)

contamination_rate = 15 / 265

iso = IsolationForest(
    random_state=42,
    contamination=contamination_rate
)

behaviour["Anomaly"] = iso.fit_predict(features_scaled)

# -1 = anomaly
behaviour["Predicted_Anomaly"] = (
    behaviour["Anomaly"] == -1
)

seeded = behaviour[
    behaviour["txn_id"].str.startswith("BTXNA")
]

detected = seeded[
    seeded["Predicted_Anomaly"] == True
]

recall = len(detected) / len(seeded)

print("\n")
print("="*70)
print("ISOLATION FOREST RESULTS")
print("="*70)

print("Injected anomalies :", len(seeded))
print("Detected anomalies :", len(detected))
print("Recall :", round(recall * 100, 2), "%")

behaviour.to_csv(
    "txn_behaviour_with_predictions.csv",
    index=False
)

# ==========================================
# SAVE COMPARISON TABLE
# ==========================================

comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("\nComparison table saved.")
print("Risk pricing table saved.")
print("Anomaly results saved.")

# ==========================================
# PROJECT COMPLETED
# ==========================================

print("\n")
print("="*70)
print("CREDIT RISK & LENDING ML PROJECT COMPLETED")
print("="*70)



RISK-BASED PRICING TABLE
        Risk_Tier  Applicants  Avg_Default_Probability  Observed_Default_Rate  \
0        Low Risk          25                     2.01                    8.0   
1     Medium Risk          25                     7.29                   12.0   
2       High Risk          25                    23.39                   20.0   
3  Very High Risk          25                    58.70                   40.0   

  Interest_Rate  
0     10% - 12%  
1     13% - 16%  
2     17% - 20%  
3     21% - 28%  

Transaction Dataset Shape
(265, 6)


ISOLATION FOREST RESULTS
Injected anomalies : 15
Detected anomalies : 11
Recall : 73.33 %

Comparison table saved.
Risk pricing table saved.
Anomaly results saved.


CREDIT RISK & LENDING ML PROJECT COMPLETED
